In [20]:
import os, pandas as pd


In [21]:
df = pd.read_csv(r'C:\Users\HP\OneDrive\Desktop\KPMG sem3\hyperparameter tuning\Hr-Attrition-V2.csv')

In [22]:
df.head()

,Employee_ID,Age,Monthly_Income,Years_at_Company,Job_Level,Overtime,Job_Satisfaction,Work_Life_Balance,Performance_Score,Distance_from_Home_km,...,Late_Days_Last_3_Months,Absence_Days_Last_Year,Commute_Time_Minutes,Employee_Engagement_Score,Manager_Rating,Internal_Transfers,Remote_Work_Days_Per_Week,Certifications,Team_Size,Attrition
0,EMP00001,55,85264.0,2.4,1,No,4,5,4.1,20,...,1,0,43.0,2.4,5.0,0,5,0,10,1
1,EMP00002,28,72037.0,2.8,2,Yes,4,2,2.7,12,...,2,3,76.0,3.4,3.8,0,0,1,16,0
2,EMP00003,22,92504.0,3.7,4,No,1,3,3.4,23,...,3,2,64.0,2.9,1.8,0,5,1,16,1
3,EMP00004,46,42949.0,2.5,1,No,5,3,3.3,21,...,4,4,58.0,2.6,3.6,0,5,2,11,1
4,EMP00005,35,25649.0,15.0,5,No,2,1,1.9,7,...,2,3,44.0,3.7,3.6,0,1,0,9,0


In [23]:
df['Attrition'].value_counts()

Attrition
1    6072
0    5928
Name: count, dtype: int64

In [24]:
X = df.drop(columns=["Employee_ID", "Attrition"])
Y = df["Attrition"]

In [25]:
from sklearn.model_selection import train_test_split, GridSearchCV
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.30,
    random_state=42,
    stratify=Y
)

In [26]:
print(X_train.shape)
print(Y_train.shape)

(8400, 29)
(8400,)


In [27]:
numeric_column = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_column = X_train.select_dtypes(include=["object"]).columns

C:\Users\HP\AppData\Local\Temp\ipykernel_4980\618520444.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_column = X_train.select_dtypes(include=["object"]).columns


In [28]:
print("Numeric columns are: \n ",numeric_column)
print(f'\ncategorical cols are :\n {categorical_column}')

Numeric columns are: 
  Index(['Age', 'Monthly_Income', 'Years_at_Company', 'Job_Level',
       'Job_Satisfaction', 'Work_Life_Balance', 'Performance_Score',
       'Distance_from_Home_km', 'Training_Hours_Last_Year',
       'Promotions_Last_5_Years', 'Salary_Hike_Percent',
       'Monthly_Working_Hours', 'Projects_Handled', 'Late_Days_Last_3_Months',
       'Absence_Days_Last_Year', 'Commute_Time_Minutes',
       'Employee_Engagement_Score', 'Manager_Rating', 'Internal_Transfers',
       'Remote_Work_Days_Per_Week', 'Certifications', 'Team_Size'],
      dtype='str')

categorical cols are :
 Index(['Overtime', 'Stock_Option', 'Business_Travel', 'Department',
       'Education_Level', 'Job_Role', 'Marital_Status'],
      dtype='str')


In [29]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_pipeline=Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [30]:
categorical_pipeline =Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [31]:
# combine preprocessinng

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_column),
    ("categorical", categorical_pipeline, categorical_column)
])

In [32]:
from sklearn.linear_model import LogisticRegression

In [33]:
# from sklearn.impute import SimpleImputer

model_before =Pipeline([
    ("preprocessing", preprocessor),
    ('model', LogisticRegression(
        C=0.00001,
        max_iter=2000
    ))

])

In [34]:
model_before.fit(X_train, Y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tr

In [35]:
y_pred_before = model_before.predict(X_test)

In [36]:
from sklearn.metrics import accuracy_score, precision_score, f1_score

accuracy_before = accuracy_score(Y_test, y_pred_before)
precision_before = precision_score(Y_test, y_pred_before, zero_division=0)
f1_score_before = f1_score(Y_test, y_pred_before, zero_division=0)


print("Accuracy:", accuracy_before)
print("Precision:", precision_before)
print("F1-score:", f1_score_before)


Accuracy: 0.5094444444444445
Precision: 0.5078212290502794
F1-score: 0.6730840429470566


# Hyperparameter

In [40]:
param_grid = {
    "model__C": [0.001, 0.01, 1, 10],
    "model__solver": ["liblinear", "lbfgs"],
    "model__class_weight": [None, "balanced"]
}

grid_search = GridSearchCV(
    model_before,
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid_search.fit(X_train, Y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._iter=2000))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.001, 0.01, ...], 'model__class_weight': [None, 'balanced'], 'model__solver': ['liblinear', 'lbfgs']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and paramete